# TumorTrace — Training Notebook

This notebook is a thin Colab/Jupyter wrapper around the repo's Python modules. It does **not** re-implement any logic — every cell below calls into `preprocess.py`, `dataset.py`, `model.py`, and `train.py` so the notebook and `train.py` can never drift apart. `train.py` is the source of truth; run it directly instead of this notebook if you prefer a plain script.

Designed to run on a free-tier Colab T4 GPU, but also runs on a local Jupyter kernel (the Colab-only cells below detect and skip themselves automatically).

## 1. Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print(f"Running in Colab: {IN_COLAB}")

In [ ]:
!pip install -q segmentation-models-pytorch albumentations monai nibabel kaggle
# torch/torchvision ship preinstalled on Colab runtimes; if running locally and you
# already have a venv with requirements.txt installed, this cell is a harmless no-op.

In [ ]:
if IN_COLAB:
    !git clone https://github.com/<your-username>/tumortrace.git
    %cd tumortrace
else:
    print("Not in Colab -- assuming this notebook is already running from inside the repo.")

## 2. Download the BraTS 2020 dataset

Requires a Kaggle account + API token (from Kaggle account settings -> "Create New API Token"). Kaggle now issues a plain bearer token rather than the older `kaggle.json` key file; either place it at `~/.kaggle/access_token` beforehand (works both in Colab and locally, and is what the cell below checks for first) or, if running in Colab, upload it interactively when prompted. See the README's "Run it yourself" section for the Medical Segmentation Decathlon fallback if Kaggle access is unavailable.

In [ ]:
import os

token_path = os.path.expanduser("~/.kaggle/access_token")
if not os.path.exists(token_path):
    if IN_COLAB:
        from google.colab import files
        os.makedirs(os.path.dirname(token_path), exist_ok=True)
        print("Paste your Kaggle API token into the upload prompt (as a plain text file), or upload kaggle.json.")
        uploaded = files.upload()
        for fname in uploaded:
            os.rename(fname, token_path)
        os.chmod(token_path, 0o600)
    else:
        raise SystemExit(
            f"No Kaggle credentials found at {token_path}. Generate a token at "
            "kaggle.com/settings/api and save it there before re-running this cell."
        )
else:
    print(f"Found Kaggle credentials at {token_path}")

In [ ]:
!kaggle datasets download -d awsaf49/brats20-dataset-training-validation
!unzip -q brats20-dataset-training-validation.zip -d data/raw

## 3. Preprocess: NIfTI volumes -> 2D slice pairs

In [ ]:
import preprocess

# The Kaggle zip extracts training and (unlabeled) validation data as siblings:
#   data/raw/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_XXX/...
#   data/raw/BraTS2020_ValidationData/MICCAI_BraTS2020_ValidationData/BraTS20_Validation_XXX/...
# Point at the TrainingData branch specifically -- the validation patients have no
# segmentation labels, so they can't be used here (preprocess.py skips and logs any
# patient it can't process rather than aborting, but there's no reason to attempt them).
# preprocess.discover_patients() also auto-detects nested wrapper folders on its own,
# so pointing this at the plain "data/raw" would still work, just slower.
raw_dir = "data/raw/BraTS2020_TrainingData"
out_dir = "data/processed"
os.makedirs(out_dir, exist_ok=True)

patients = preprocess.discover_patients(raw_dir)
print(f"Found {len(patients)} patients")

import json
import numpy as np

splits = preprocess.split_patients(list(patients.keys()))
with open(os.path.join(out_dir, "split_patients.json"), "w") as f:
    json.dump(splits, f, indent=2)

rng = np.random.RandomState(42)
total = 0
failed = []
for patient_id, patient_dir in patients.items():
    try:
        total += preprocess.process_patient(patient_id, patient_dir, out_dir, rng)
    except Exception as e:
        failed.append(patient_id)
        print(f"SKIPPED {patient_id}: {e}")
print(f"Saved {total} slices to {out_dir} ({len(patients) - len(failed)}/{len(patients)} patients)")

## 4. Train

In [ ]:
from train import train_model

history, best_val_wt_dice = train_model(processed_dir="data/processed")
print(f"Best val WT Dice: {best_val_wt_dice:.4f}")

## 5. Evaluate on the held-out test set

In [ ]:
!python evaluate.py --raw_dir data/raw/BraTS2020_TrainingData --processed_dir data/processed --checkpoint checkpoints/best_model.pt

`evaluate.py` writes `results/metrics_table.md` and `results/qualitative_examples.png`. Commit both plus `checkpoints/best_model.pt` back to the repo so `app.py` has a trained model to serve. Run `python make_samples.py` afterward to refresh the bundled demo cases in `samples/` from the new checkpoint's test split.